In [2]:
pip install filelock

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import herbie
import xarray as xr
import zarr
from filelock import FileLock

print(f"Python: {sys.version}")
print(f"herbie: {herbie.__version__}")
print(f"xarray: {xr.__version__}")
print(f"zarr: {zarr.__version__}")

# Test URMA access
from herbie import Herbie
H = Herbie('2024-01-01', model='urma')
print("\n✓ STACK READY!!!!")

Python: 3.14.3 | packaged by conda-forge | (main, Feb  9 2026, 22:09:14) [Clang 20.1.8 ]
herbie: 2026.1.1
xarray: 2025.11.0
zarr: 3.1.5
✅ Found ┊ model=urma ┊ product=anl ┊ 2024-Jan-01 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ None

✓ STACK READY!!!!


In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [4]:
import xarray as xr
xr.set_options(use_new_combine_kwarg_defaults=True)


In [23]:
from pathlib import Path
import shutil

BASE_CACHE = Path("/Users/amysteward/Desktop/MIDS/210/herbie_cache")

def day_cache_dir(date_val):
    d = pd.to_datetime(date_val).strftime("%Y%m%d")
    p = BASE_CACHE / d
    p.mkdir(parents=True, exist_ok=True)
    return str(p)

In [24]:
VARIABLE_CONFIG = {
    'tmax': {
        'vars': ['TMAX:2 m above ground'],  # Simple strings
        'times': [8],  # 8 UTC = TMAX analysis, 20 UTC = TMIN analysis
        'output_names': ['t2m_max_k'],
        'description': 'Temperature max at 2m (URMA daily extremes)'
    },
    'tmin': {
        'vars': ['TMIN:2 m above ground'],  # Simple strings
        'times': [20],  # 8 UTC = TMAX analysis, 20 UTC = TMIN analysis
        'output_names': ['t2m_min_k'],
        'description': 'Temperature min at 2m (URMA daily extremes)'
    },
    'dew_point': {
        'vars': ['DPT:2 m above ground'],
        'times': [22, 14],  # Same as temperature for humidity calculations
        'output_names': ['dpt_afternoon_k', 'dpt_morning_k'],
        'description': 'Dew point at 2m (for relative humidity calculation)'
    },
    'cloud_cover': {
        'vars': ['TCDC:entire atmosphere'],
        'times': [20],  # 20 UTC = 12-1 PM PT (peak solar)
        'output_names': ['cloud_cover_pct'],
        'description': 'Total cloud cover percentage (peak solar hours)'
    },
    'wind': {
        'vars': ['WIND:10 m above ground'],
        'times': [22, 8],  # 22 UTC = 2-3 PM PT (peak demand), 8 UTC = 12-1 AM PT (low demand)
        'output_names': ['wind_peak_ms', 'wind_low_ms'],
        'description': 'Wind speed at 10m (peak demand and low demand periods)'
    },
    'specific_humidity': {
        'vars': ['SPFH:2 m above ground'],
        'times': [22],  # 22 UTC = 2-3 PM PT (peak A/C load)
        'output_names': ['spfh_peak_kgkg'],
        'description': 'Specific humidity at 2m (peak A/C period)'
    }
}

In [25]:
DROP_COORDS = ["time", "valid_time", "step", "heightAboveGround"]

def to_daily_feature_date(da: xr.DataArray, date_str: str) -> xr.DataArray:
    """
    Convert a raw URMA DataArray into a daily feature:
      - removes merge-breaking dims/coords (time/valid_time/step/heightAboveGround)
      - adds a single 'date' dimension (UTC midnight for date_str)
    """
    # If any of these appear as dimensions (rare but possible), squeeze them out
    for d in DROP_COORDS:
        if d in da.dims:
            da = da.isel({d: 0}).squeeze(drop=True)

    # Drop merge-breaking coords so xr.Dataset doesn't try to align on them
    drop_these = [c for c in DROP_COORDS if c in da.coords]
    if drop_these:
        da = da.reset_coords(drop_these, drop=True)

    # Add a single daily index (UTC day label)
    date_coord = pd.to_datetime(date_str).normalize()
    return da.expand_dims("date").assign_coords(date=[date_coord])


In [26]:
from collections import defaultdict
from herbie import Herbie
import pandas as pd
import xarray as xr
import numpy as np

def build_day_plan(date_str, selected_variables):
    """
    Returns: dict[datetime] -> list of (raw_search, output_name)
    """
    plan = defaultdict(list)
    base = pd.to_datetime(date_str)

    for var_key in selected_variables:
        cfg = VARIABLE_CONFIG[var_key]
        raw_search = cfg["vars"][0]
        for hour, out_name in zip(cfg["times"], cfg["output_names"]):
            dt = base + pd.Timedelta(hours=int(hour))
            plan[dt].append((raw_search, out_name))

    return dict(plan)

In [27]:
from filelock import FileLock
from pathlib import Path
import shutil
import xarray as xr
import numpy as np

def verify_day_dataset(ds_check, test=(5,5)):
    broken = {}
    for v in ds_check.data_vars:
        da = ds_check[v]
        try:
            sel = {}
            if "date" in da.dims: sel["date"] = 0
            if "y" in da.dims: sel["y"] = slice(0, test[0])
            if "x" in da.dims: sel["x"] = slice(0, test[1])
            da.isel(**sel).compute()
        except Exception as e:
            broken[v] = repr(e)
    return broken

def append_verified_day(day_dataset, zarr_path, date_str, verbose=True):
    # 1) write day to temp store
    tmp_store = f"{zarr_path}.tmp_{date_str.replace('-','')}.zarr"
    if Path(tmp_store).exists():
        shutil.rmtree(tmp_store, ignore_errors=True)

    day_dataset.to_zarr(tmp_store, mode="w", consolidated=False)

    # 2) verify temp store by re-opening it (catches codec/write issues)
    ds_tmp = xr.open_zarr(tmp_store, consolidated=False)
    broken = verify_day_dataset(ds_tmp)
    if broken:
        shutil.rmtree(tmp_store, ignore_errors=True)
        raise RuntimeError(f"Temp day store corrupt for {date_str}: {broken}")

    # 3) append into the main store under a filesystem lock
    lock = FileLock(zarr_path + ".lock")
    with lock:
        if not Path(zarr_path).exists():
            ds_tmp.to_zarr(zarr_path, mode="w", consolidated=False)
        else:
            ds_tmp.to_zarr(zarr_path, mode="a", append_dim="date", consolidated=False)

    # 4) cleanup temp
    shutil.rmtree(tmp_store, ignore_errors=True)

    if verbose:
        print(f"  ✅ Verified+appended {date_str}")


In [28]:
def _clear_cached_index(H):
    # index_as_dataframe is a functools.cached_property; clear its cached value if present
    H.__dict__.pop("index_as_dataframe", None)

def _force_local_index(H):
    """
    After downloading the full GRIB, force Herbie to generate an IDX using wgrib2
    instead of trying the remote NOMADS .idx URL.
    """
    H.download()                 # ensure local GRIB exists
    H.grib_source = "local"      # critical: tells index_as_dataframe to use wgrib2
    H.idx = None                 # critical: prevents requests.get(H.idx) to remote
    _clear_cached_index(H)



In [29]:
from __future__ import annotations

import os
import warnings
from typing import Any, Dict, List, Optional, Union

import xarray as xr
from herbie import Herbie


# Type aliases for better readability
ReqSpec = Union[
    str,                              # Combined search string: "TMP:2 m|DPT:2 m"
    List[str],                        # List of search strings (joined with |)
    Dict[str, Union[str, List[str]]]  # Named requests: {"temp": "TMP:2 m", "dew": ["DPT:2 m"]}
]


def normalize_search_requests(requests: Optional[ReqSpec]) -> Dict[str, str]:
    """
    Normalize different request formats into a consistent dict format.
    
    Args:
        requests: Search specification in various formats
        
    Returns:
        Dict mapping request names to search strings
        
    Examples:
        normalize_search_requests("TMP:2 m|DPT:2 m") 
        -> {"__all__": "TMP:2 m|DPT:2 m"}
        
        normalize_search_requests(["TMP:2 m", "DPT:2 m"])
        -> {"__all__": "TMP:2 m|DPT:2 m"}
        
        normalize_search_requests({"temp": "TMP:2 m", "dew": "DPT:2 m"})
        -> {"temp": "TMP:2 m", "dew": "DPT:2 m"}
    """
    if requests is None:
        return {"__all__": ""}

    if isinstance(requests, str):
        return {"__all__": requests}

    if isinstance(requests, list):
        return {"__all__": "|".join(requests)}

    if isinstance(requests, dict):
        normalized = {}
        for name, search_spec in requests.items():
            if isinstance(search_spec, str):
                normalized[name] = search_spec
            elif isinstance(search_spec, list):
                normalized[name] = "|".join(search_spec)
            else:
                raise TypeError(
                    f"Request '{name}' must be str or list[str], got {type(search_spec)}"
                )
        return normalized

    raise TypeError(f"Requests must be str, list[str], dict, or None. Got {type(requests)}")


def load_herbie_with_fallback(
    herbie_obj: Herbie,
    search_string: str = "",
    engine: str = "cfgrib",
    verbose: bool = False,
    **xarray_kwargs: Any,
) -> Union[xr.Dataset, List[xr.Dataset]]:
    """
    Load data from Herbie object with automatic fallback for missing index files.
    
    This function handles the common case where URMA files on AWS don't have
    .idx sidecar files, requiring a fallback to download the full GRIB2 file.
    
    Args:
        herbie_obj: Herbie object to load data from
        search_string: GRIB search pattern (empty string loads all variables)
        engine: xarray engine for opening files
        verbose: Whether to print status messages
        **xarray_kwargs: Additional arguments passed to xarray
        
    Returns:
        Dataset or list of datasets (when cfgrib creates multiple hypercubes)
        
    Raises:
        ValueError: For GRIB loading errors that can't be resolved by fallback
        RuntimeError: If fallback download fails
    """
    try:
        # First attempt: use search string if provided
        if search_string.strip():
            return herbie_obj.xarray(search=search_string, engine=engine, **xarray_kwargs)
        else:
            return herbie_obj.xarray(engine=engine, **xarray_kwargs)

    except ValueError as e:
        error_msg = str(e)
        
        # Check if this is the missing index file error we can handle
        if "No index file was found for" not in error_msg:
            raise  # Re-raise other ValueError types
            
        if verbose:
            print(f"⚠️  Missing .idx file - downloading full GRIB2 and retrying...")

        try:
            # CRITICAL FIX: Download without passing engine parameter
            herbie_obj.download()
            
            # Create fresh Herbie object to clear any cached index properties
            fresh_herbie = Herbie(
                date=herbie_obj.date,
                model=herbie_obj.model,
                product=herbie_obj.product,
                fxx=herbie_obj.fxx,
                save_dir=getattr(herbie_obj, "save_dir", None),
                source=getattr(herbie_obj, "source", None),
            )
            
            # Open full file without search filtering (since no .idx available)
            # Engine parameter goes HERE, not in download()
            return fresh_herbie.xarray(engine=engine, **xarray_kwargs)
            
        except Exception as download_error:
            raise RuntimeError(
                f"Fallback download failed for {herbie_obj.date}: {download_error}"
            ) from download_error


def load_herbie_datasets_safe(
    herbie_objects: Union[Herbie, List[Herbie]],
    search_requests: Optional[ReqSpec] = None,
    engine: str = "cfgrib",
    remove_grib_after: bool = False,
    verbose: bool = False,
    **xarray_kwargs: Any
) -> Dict[str, Any]:
    """
    Safely load datasets from one or more Herbie objects with comprehensive error handling.
    
    This function provides robust loading of URMA data with:
    - Automatic fallback for missing .idx files
    - Individual error tracking per Herbie object
    - Optional GRIB file cleanup
    - Support for multiple search request formats
    
    Args:
        herbie_objects: Single Herbie object or list of Herbie objects
        search_requests: Variables to extract (see ReqSpec type for formats)
        engine: xarray engine for reading GRIB files
        remove_grib_after: Whether to delete local GRIB files after loading
        verbose: Whether to print detailed status messages
        **xarray_kwargs: Additional arguments passed to xarray
        
    Returns:
        Dictionary with:
            - "datasets": List of successfully loaded xr.Dataset objects
            - "failures": List of error information for failed loads
            - "requested": Normalized search requests that were attempted
            
    Example:
        >>> herbie_list = [Herbie('2024-01-01', model='urma'), ...]
        >>> result = load_herbie_datasets_safe(
        ...     herbie_list, 
        ...     search_requests="TMP:2 m above ground|DPT:2 m above ground"
        ... )
        >>> datasets = result["datasets"]
        >>> failures = result["failures"]
    """
    # Normalize inputs
    normalized_requests = normalize_search_requests(search_requests)
    herbie_list = herbie_objects if isinstance(herbie_objects, list) else [herbie_objects]
    
    # Track results
    datasets: List[xr.Dataset] = []
    failures: List[Dict[str, Any]] = []

    for i, herbie_obj in enumerate(herbie_list):
        try:
            loaded_datasets = []
            
            if "__all__" in normalized_requests:
                # Single combined search
                search_string = normalized_requests["__all__"].strip()
                
                result = load_herbie_with_fallback(
                    herbie_obj=herbie_obj,
                    search_string=search_string,
                    engine=engine,
                    verbose=verbose,
                    **xarray_kwargs
                )
                
                # Handle both single dataset and list of datasets
                if isinstance(result, list):
                    loaded_datasets.extend(result)
                else:
                    loaded_datasets.append(result)
                    
            else:
                # Multiple named searches - load each and merge
                dataset_parts: List[xr.Dataset] = []
                
                for request_name, search_string in normalized_requests.items():
                    search_string = search_string.strip()
                    
                    result = load_herbie_with_fallback(
                        herbie_obj=herbie_obj,
                        search_string=search_string,
                        engine=engine,
                        verbose=verbose,
                        **xarray_kwargs
                    )
                    
                    if isinstance(result, list):
                        dataset_parts.extend(result)
                    else:
                        dataset_parts.append(result)
                
                # Merge all parts into single dataset
                if len(dataset_parts) == 1:
                    loaded_datasets.append(dataset_parts[0])
                else:
                    merged = xr.merge(dataset_parts, compat="no_conflicts", join="outer")
                    loaded_datasets.append(merged)
            
            # Add successfully loaded datasets to results
            datasets.extend(loaded_datasets)
            
            # Optional cleanup of local GRIB files
            if remove_grib_after:
                _cleanup_grib_file(herbie_obj, verbose=verbose)

        except Exception as e:
            # Record detailed failure information
            failure_info = {
                "index": i,
                "error": str(e),
                "error_type": type(e).__name__,
                "herbie_model": getattr(herbie_obj, "model", None),
                "herbie_product": getattr(herbie_obj, "product", None), 
                "herbie_date": getattr(herbie_obj, "date", None),
                "herbie_fxx": getattr(herbie_obj, "fxx", None),
                "herbie_source": getattr(herbie_obj, "source", None),
            }
            failures.append(failure_info)
            
            if verbose:
                date_str = getattr(herbie_obj, "date", "unknown")
                fxx = getattr(herbie_obj, "fxx", "unknown")
                print(f"❌ Failed to load {date_str} f{fxx}: {e}")

    return {
        "datasets": datasets, 
        "failures": failures, 
        "requested": normalized_requests
    }


def _cleanup_grib_file(herbie_obj: Herbie, verbose: bool = False) -> None:
    """
    Helper function to safely remove local GRIB files after processing.
    
    Args:
        herbie_obj: Herbie object that may have downloaded local files
        verbose: Whether to print cleanup status messages
    """
    try:
        local_file = getattr(herbie_obj, "local_file", None)
        if local_file and os.path.exists(local_file):
            os.remove(local_file)
            if verbose:
                print(f"🧹 Cleaned up: {os.path.basename(local_file)}")
    except Exception as e:
        if verbose:
            print(f"⚠️  Cleanup failed for {getattr(herbie_obj, 'date', 'unknown')}: {e}")


# Backwards compatibility aliases - YOUR EXISTING CODE WILL STILL WORK
_normalize_reqs = normalize_search_requests
_open_with_search_or_fallback = load_herbie_with_fallback
herbie_xarray_list_safe = load_herbie_datasets_safe

In [30]:
def get_var_from_cubes(cubes, varname):
    target = varname.lower()

    # try exact match (case-insensitive)
    for ds in cubes:
        for v in ds.data_vars:
            if v.lower() == target:
                return ds[v]

    # fallback: some datasets use aliases; print helpful info
    available = [list(ds.data_vars) for ds in cubes]
    raise KeyError(f"{varname} not found in any hypercube. Available: {available}")


In [31]:
import numpy as np
import xarray as xr

EXPECTED_VARS = [
    "cloud_cover_pct",
    "dpt_afternoon_k",
    "dpt_morning_k",
    "spfh_peak_kgkg",
    "t2m_max_k",
    "t2m_min_k",
    "wind_low_ms",
    "wind_peak_ms",
]

def ensure_all_vars_present(day_ds: xr.Dataset, expected=EXPECTED_VARS) -> xr.Dataset:
    """
    Ensure every expected var exists in day_ds with dims (date,y,x).
    Missing vars are filled with NaNs (float32) for that date.
    """
    # pick a template 3D variable to get y/x coords + shape
    template_name = next(v for v in day_ds.data_vars if day_ds[v].ndim == 3)
    tmpl = day_ds[template_name]
    ny, nx = tmpl.shape[-2], tmpl.shape[-1]

    for v in expected:
        if v not in day_ds:
            day_ds[v] = xr.DataArray(
                np.full((1, ny, nx), np.nan, dtype="float32"),
                dims=("date", "y", "x"),
                coords={"date": day_ds["date"], "y": tmpl["y"], "x": tmpl["x"]},
                name=v,
            )
    return day_ds

In [32]:
from concurrent.futures import ThreadPoolExecutor, as_completed

RAW_TO_DVAR = {
    "WIND:10 m above ground": "si10",
    "DPT:2 m above ground": "d2m",
    "SPFH:2 m above ground": "sh2",
    "TCDC:entire atmosphere": "tcc",
    "TMAX:2 m above ground": "tmax",
    "TMIN:2 m above ground": "tmin",
}

def _load_one_dt(dt, requests, save_dir, ca_window):
    H = Herbie(dt, model="urma", product="anl", fxx=0, save_dir=save_dir)
    combined_search = "|".join(sorted({s for s, _ in requests}))

    # Use the FIXED function
    out_load = load_herbie_datasets_safe(H, combined_search, verbose=True)  # FIXED VERSION
    cubes = out_load["datasets"]          # ✅ LIST of datasets

    if not cubes:
        # keep the error informative
        raise RuntimeError(f"No cubes opened for {dt}. failures={out_load['failures'][:2]}")

    if ca_window is None:
        ca_window_local = compute_ca_window(cubes[0])
    else:
        ca_window_local = ca_window

    ysl, xsl = ca_window_local
    cubes = [ds.isel(y=ysl, x=xsl) for ds in cubes]

    out = {}
    for raw_search, out_name in requests:
        dvar = RAW_TO_DVAR[raw_search]
        out[out_name] = get_var_from_cubes(cubes, dvar)
    return dt, out, ca_window_local


def extract_day_from_plan_fast(date_str, selected_variables, ca_window=None, save_dir=None, max_workers=4):
    plan = build_day_plan(date_str, selected_variables)

    # parallel load 4 timestamps
    results = {}
    ca_window_out = ca_window

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [
            ex.submit(_load_one_dt, dt, reqs, save_dir, ca_window_out)
            for dt, reqs in sorted(plan.items())
        ]
        for fut in as_completed(futs):
            dt, out_dt, caw = fut.result()
            results.update(out_dt)
            if ca_window_out is None:
                ca_window_out = caw

    return results, ca_window_out



In [33]:
def process_urma_to_zarr_enhanced(
    start_date: str,
    end_date: str,
    zarr_path: str,
    selected_variables: Optional[List[str]] = None,
    chunk_size: Union[str, Dict, None] = "auto",
    save_dir: Optional[str] = None,
    ca_only: bool = True,
    cleanup_cache: bool = True,
    verbose: bool = True
) -> tuple[xr.Dataset, List[str]]:
    """
    Process URMA data into a consolidated Zarr store with robust error handling.
    
    Args:
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format (inclusive)
        zarr_path: Output path for the Zarr store
        selected_variables: List of variables to process (uses default set if None)
        chunk_size: Chunking strategy - "auto" (xarray autochunk), "manual_date" (date=1), 
                    dict with dimension:size pairs, or None/"none" for no chunking
        save_dir: Base directory for temporary cache files (uses default if None)
        ca_only: Whether to use consistent California-only spatial window
        cleanup_cache: Whether to delete temporary cache directories after processing
        verbose: Whether to print detailed progress messages
        
    Returns:
        Tuple of (final_dataset, list_of_failed_dates)
        
    Raises:
        RuntimeError: If no dates process successfully
        ValueError: If date range or parameters are invalid
    """
    import pandas as pd
    import xarray as xr
    import zarr
    from pathlib import Path
    import shutil
    
    # Default variable set for comprehensive weather analysis
    if selected_variables is None:
        selected_variables = [
            "tmax", "tmin", "dew_point", 
            "cloud_cover", "wind", "specific_humidity"
        ]
    
    # Validate inputs
    try:
        date_range = pd.date_range(start_date, end_date, freq="D")
    except Exception as e:
        raise ValueError(f"Invalid date range '{start_date}' to '{end_date}': {e}")
    
    if len(date_range) == 0:
        raise ValueError("Date range is empty")
    
    # Initialize tracking variables
    failed_dates = []
    ca_window = None  # Spatial window for California region
    successfully_wrote = False
    total_dates = len(date_range)
    
    if verbose:
        print(f"🚀 Starting URMA processing: {total_dates} dates from {start_date} to {end_date}")
        print(f"📊 Variables: {', '.join(selected_variables)}")
        print(f"💾 Output: {zarr_path}")
    
    # Process each date
    for i, date in enumerate(date_range):
        date_str = date.strftime("%Y-%m-%d")
        progress = f"{i+1}/{total_dates}"
        
        if verbose:
            print(f"\n📅 Processing {progress}: {date_str}")
        
        # Set up temporary cache directory for this date
        try:
            cache_dir = day_cache_dir(date_str) if save_dir is None else str(Path(save_dir) / date_str.replace('-', ''))
            Path(cache_dir).mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  ❌ Failed to create cache directory: {e}")
            failed_dates.append(date_str)
            continue
        
        try:
            # Extract data for this day across all required timestamps
            day_data, ca_window = extract_day_from_plan_fast(
                date_str=date_str,
                selected_variables=selected_variables,
                ca_window=ca_window if ca_only else None,  # Reuse spatial window for consistency
                save_dir=cache_dir,
                max_workers=4  # Parallel processing of timestamps
            )
            
            # Validate that we got some data
            if not day_data or len(day_data) == 0:
                raise RuntimeError("No variables extracted - all timestamps may have failed")
            
            if verbose:
                print(f"  📊 Extracted {len(day_data)} variables")
            
            # Convert raw DataArrays to daily features with consistent time coordinate
            try:
                processed_vars = {}
                for var_name, data_array in day_data.items():
                    processed_vars[var_name] = to_daily_feature_date(data_array, date_str)
                
                day_dataset = xr.Dataset(processed_vars)
                
                # Ensure all expected variables are present (fill missing with NaNs)
                day_dataset = ensure_all_vars_present(day_dataset)
                
            except Exception as e:
                raise RuntimeError(f"Failed to process variables into dataset: {e}")
            
            # Apply chunking strategy
            try:
                if chunk_size == "auto":
                    # Use xarray's automatic chunking based on data size and memory constraints
                    day_dataset = day_dataset.chunk("auto")
                elif chunk_size == "manual_date":
                    # Fallback: chunk only the date dimension manually
                    day_dataset = day_dataset.chunk({"date": 1})
                elif isinstance(chunk_size, dict):
                    # Custom chunking scheme
                    day_dataset = day_dataset.chunk(chunk_size)
                elif chunk_size is None or chunk_size == "none":
                    # Leave unchunked
                    pass
                else:
                    print(f"  ⚠️ Warning: Unknown chunk_size '{chunk_size}', using auto")
                    day_dataset = day_dataset.chunk("auto")
                    
            except Exception as e:
                print(f"  ⚠️ Warning: Chunking failed, proceeding unchunked: {e}")
            
            # Handle static coordinates (only write once, then drop for subsequent dates)
            static_coords = ["latitude", "longitude", "atmosphereSingleLayer", "gribfile_projection"]
            
            if successfully_wrote:
                # Drop static coordinates for append operations (they're already in the Zarr)
                coords_to_drop = [coord for coord in static_coords if coord in day_dataset.variables]
                if coords_to_drop:
                    day_dataset = day_dataset.drop_vars(coords_to_drop, errors="ignore")

            # Write day to temp, verify, then append into main store (lock-protected)
            try:
                append_verified_day(day_dataset, zarr_path, date_str, verbose=verbose)
                successfully_wrote = True  # only after verified append succeeds
            
                if verbose:
                    total_vars = len(day_dataset.data_vars)
                    print(f"  ✅ Wrote {date_str} ({len(day_data)} raw → {total_vars} final variables)")
            
            except Exception as e:
                raise RuntimeError(f"Failed to write verified day: {e}")
            

            
            # # Write to Zarr store
            # try:
            #     write_mode = "w" if not successfully_wrote else "a"
            #     append_dimension = None if not successfully_wrote else "date"
            #     lock = FileLock(zarr_path + ".lock")  # creates a tiny lockfile next to the store
            #     with lock:
            #         day_dataset.to_zarr(
            #             zarr_path,
            #             mode=write_mode,
            #             append_dim=append_dimension,
            #             consolidated=False  # We'll consolidate at the end
            #         )
                
                
            #     date_value = day_dataset["date"].values[0]
            #     broken = verify_written_day(zarr_path, date_value, list(day_dataset.data_vars))
            #     if broken:
            #         raise RuntimeError(f"Corrupt write for {date_value}: {broken}")
            #     successfully_wrote = True
            #     if verbose:
            #         total_vars = len(day_dataset.data_vars)
            #         print(f"  ✅ Wrote {date_str} ({len(day_data)} raw → {total_vars} final variables)")
                
            except Exception as e:
                raise RuntimeError(f"Failed to write to Zarr store: {e}")
            
            # Clean up temporary cache
            if cleanup_cache:
                try:
                    shutil.rmtree(cache_dir, ignore_errors=True)
                    if verbose:
                        print(f"  🧹 Cleaned up cache: {Path(cache_dir).name}")
                except Exception as e:
                    print(f"  ⚠️ Cache cleanup failed: {e}")
        
        except Exception as e:
            error_msg = str(e)
            print(f"  ❌ Failed {date_str}: {error_msg}")
            failed_dates.append(date_str)
            
            # Clean up cache even on failure
            if cleanup_cache:
                try:
                    shutil.rmtree(cache_dir, ignore_errors=True)
                except:
                    pass  # Ignore cleanup errors on failed processing
    
    # Final validation and consolidation
    if not successfully_wrote:
        failed_summary = f"All {len(failed_dates)} dates failed"
        raise RuntimeError(f"No dates processed successfully. {failed_summary}: {failed_dates}")
    
    # Consolidate Zarr metadata for better performance
    try:
        zarr.consolidate_metadata(zarr_path)
        if verbose:
            print(f"\n🎯 Consolidated Zarr metadata")
    except Exception as e:
        print(f"⚠️ Warning: Failed to consolidate Zarr metadata: {e}")
    
    # Load and return final dataset
    try:
        final_dataset = xr.open_zarr(zarr_path, consolidated=True)
        
        if verbose:
            success_count = total_dates - len(failed_dates)
            print(f"\n🏁 Processing complete!")
            print(f"✅ Successfully processed: {success_count}/{total_dates} dates")
            if failed_dates:
                print(f"❌ Failed dates ({len(failed_dates)}): {failed_dates}")
            
            # Show final dataset info
            print(f"📊 Final dataset shape: {dict(final_dataset.sizes)}")
            print(f"🗂️ Variables: {list(final_dataset.data_vars.keys())}")
        
        return final_dataset, failed_dates
        
    except Exception as e:
        raise RuntimeError(f"Failed to load final Zarr dataset: {e}")
    finally:
        if cleanup_cache:
            shutil.rmtree(cache_dir, ignore_errors=True)
            if verbose:
                print(f"  🧹 Cleaned up cache: {Path(cache_dir).name}")
    

# Helper function to validate the processing results
def validate_urma_zarr(zarr_path: str, expected_variables: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    Validate a processed URMA Zarr dataset and return summary statistics.
    
    Args:
        zarr_path: Path to the Zarr store
        expected_variables: List of variables that should be present
        
    Returns:
        Dictionary with validation results and statistics
    """
    try:
        ds = xr.open_zarr(zarr_path, consolidated=True)
        
        validation_results = {
            "valid": True,
            "total_dates": len(ds.date) if "date" in ds.dims else 0,
            "date_range": (str(ds.date.min().values), str(ds.date.max().values)) if "date" in ds.dims else None,
            "variables": list(ds.data_vars.keys()),
            "spatial_shape": (ds.sizes.get("y", 0), ds.sizes.get("x", 0)),
            "missing_data_summary": {},
            "issues": []
        }
        
        # Check for expected variables
        if expected_variables:
            missing_vars = set(expected_variables) - set(ds.data_vars.keys())
            if missing_vars:
                validation_results["issues"].append(f"Missing variables: {list(missing_vars)}")
        
        # Check for missing data
        for var_name in ds.data_vars:
            var_data = ds[var_name]
            if hasattr(var_data, 'isnull'):
                null_count = var_data.isnull().sum().compute()
                total_count = var_data.size
                null_fraction = float(null_count / total_count) if total_count > 0 else 0
                
                validation_results["missing_data_summary"][var_name] = {
                    "null_count": int(null_count),
                    "total_count": int(total_count),
                    "null_fraction": null_fraction
                }
                
                if null_fraction > 0.5:  # More than 50% missing
                    validation_results["issues"].append(f"Variable '{var_name}' has {null_fraction:.1%} missing data")
        
        if validation_results["issues"]:
            validation_results["valid"] = False
            
        return validation_results
        
    except Exception as e:
        return {
            "valid": False,
            "error": str(e),
            "issues": [f"Failed to load dataset: {e}"]
        }

In [34]:
CA_BOUNDS = dict(
    lat_min=32.0,
    lat_max=42.5,
    lon_min=-124.5,
    lon_max=-114.0,
)

def to_360(lon):
    return lon % 360


In [35]:
def compute_ca_window(ds, bounds=CA_BOUNDS, pad=2):
    """
    Returns (y_slice, x_slice) covering CA bounding box on a 2D lat/lon grid.
    pad: extra grid cells to include around the box.
    """
    lat = ds["latitude"]
    lon = ds["longitude"]

    lon_min = to_360(bounds["lon_min"])
    lon_max = to_360(bounds["lon_max"])

    mask = (
        (lat >= bounds["lat_min"]) &
        (lat <= bounds["lat_max"]) &
        (lon >= lon_min) &
        (lon <= lon_max)
    )

    ys, xs = np.where(mask.compute().values)
    if len(ys) == 0:
        raise ValueError("CA mask empty — check bounds or lon convention.")

    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()

    ymin = max(0, ymin - pad)
    xmin = max(0, xmin - pad)
    ymax = min(ds.sizes["y"] - 1, ymax + pad)
    xmax = min(ds.sizes["x"] - 1, xmax + pad)

    return slice(ymin, ymax + 1), slice(xmin, xmax + 1)


In [42]:
%%time

YEAR = '2020'
zarr_path = f"/Users/amysteward/Berk_mids/210_cap/data/data_pipeline/01_URMA_RAW/URMA_RAW_{YEAR}.zarr"

ds_out, failed = process_urma_to_zarr_enhanced(
    start_date="2020-01-01",
    end_date="2020-12-31",
    zarr_path=zarr_path,
    selected_variables=[
        "tmax",
        "tmin",
        "dew_point",
        "cloud_cover",
        "wind",
        "specific_humidity",
    ],
    chunk_size="auto",
    save_dir=None,
    ca_only=True,
)

print("Failed days:", failed)
print(ds_out)

🚀 Starting URMA processing: 366 dates from 2020-01-01 to 2020-12-31
📊 Variables: tmax, tmin, dew_point, cloud_cover, wind, specific_humidity
💾 Output: /Users/amysteward/Berk_mids/210_cap/data/data_pipeline/01_URMA_RAW/URMA_RAW_2020.zarr

📅 Processing 1/366: 2020-01-01
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 14:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ None
⚠️  Missing .idx file - downloading full GRIB2 and retrying...
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 20:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ None
⚠️  Missing .idx file - downloading full GRIB2 and retrying...
👨🏻‍🏭 Created directory: [/Users/amysteward/Desktop/MIDS/210/herbie_cache/20200101/urma/20200101]
👨🏻‍🏭 Created directory: [/Users/amysteward/Desktop/MIDS/210/herbie_cache/20200101/urma/20200101]
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 08:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ None
⚠️  Missing .idx file - downloading full GRIB2 and retrying...
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 22:00 UTC F00 ┊ GRIB2 @ aws ┊

/Users/amysteward/miniconda3/envs/herbie/lib/python3.14/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [6] xarray.Datasets because cfgrib opened with multiple hypercubes.
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 20:00 UTC F00 ┊ GRIB2 @ local ┊ IDX @ None


/Users/amysteward/miniconda3/envs/herbie/lib/python3.14/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [7] xarray.Datasets because cfgrib opened with multiple hypercubes.
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 08:00 UTC F00 ┊ GRIB2 @ local ┊ IDX @ None
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Jan-01 14:00 UTC F00 ┊ GRIB2 @ local ┊ IDX @ None


...

Note: Returning a list of [6] xarray.Datasets because cfgrib opened with multiple hypercubes.
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Dec-31 08:00 UTC F00 ┊ GRIB2 @ local ┊ IDX @ None


/Users/amysteward/miniconda3/envs/herbie/lib/python3.14/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [7] xarray.Datasets because cfgrib opened with multiple hypercubes.
✅ Found ┊ model=urma ┊ product=anl ┊ 2020-Dec-31 20:00 UTC F00 ┊ GRIB2 @ local ┊ IDX @ None


/Users/amysteward/miniconda3/envs/herbie/lib/python3.14/site-packages/herbie/core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Note: Returning a list of [7] xarray.Datasets because cfgrib opened with multiple hypercubes.
  📊 Extracted 8 variables
  ✅ Verified+appended 2020-12-31
  ✅ Wrote 2020-12-31 (8 raw → 8 final variables)
  🧹 Cleaned up cache: 20201231

🎯 Consolidated Zarr metadata

🏁 Processing complete!
✅ Successfully processed: 352/366 dates
❌ Failed dates (14): ['2020-03-07', '2020-03-08', '2020-04-14', '2020-04-15', '2020-04-25', '2020-05-21', '2020-05-22', '2020-05-23', '2020-05-24', '2020-05-25', '2020-06-24', '2020-08-07', '2020-09-09', '2020-11-21']
📊 Final dataset shape: {'date': 352, 'y': 535, 'x': 457}
🗂️ Variables: ['cloud_cover_pct', 'dpt_afternoon_k', 'dpt_morning_k', 'gribfile_projection', 'spfh_peak_kgkg', 't2m_max_k', 't2m_min_k', 'wind_low_ms', 'wind_peak_ms']
  🧹 Cleaned up cache: 20201231
Failed days: ['2020-03-07', '2020-03-08', '2020-04-14', '2020-04-15', '2020-04-25', '2020-05-21', '2020-05-22', '2020-05-23', '2020-05-24', '2020-05-25', '2020-06-24', '2020-08-07', '2020-09-09', '20

In [43]:
ds = xr.open_zarr(zarr_path, consolidated=True)
print(ds)
print(len(ds.date))


<xarray.Dataset> Size: 3GB
Dimensions:                (date: 352, y: 535, x: 457)
Coordinates:
  * date                   (date) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
    atmosphereSingleLayer  float64 8B ...
    latitude               (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    longitude              (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    cloud_cover_pct        (date, y, x) float32 344MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_afternoon_k        (date, y, x) float32 344MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k          (date, y, x) float32 344MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    gribfile_projection    float64 8B ...
    spfh_peak_kgkg         (date, y, x) float32 344MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k              (date, y, x) float32 344MB dask.array<chun

In [39]:
ds.t2m_max_k.isel(date=[0, -1]).mean().compute()


<xarray.DataArray 't2m_max_k' ()> Size: 4B
array(287.227, dtype=float32)
Coordinates:
    atmosphereSingleLayer  float64 8B 0.0
Attributes: (12/39)
    GRIB_DxInMetres:                          2539.703
    GRIB_DyInMetres:                          2539.703
    GRIB_LaDInDegrees:                        25.0
    GRIB_Latin1InDegrees:                     25.0
    GRIB_Latin2InDegrees:                     25.0
    GRIB_LoVInDegrees:                        265.0
    ...                                       ...
    GRIB_units:                               K
    GRIB_uvRelativeToGrid:                    1
    grid_mapping:                             gribfile_projection
    long_name:                                Maximum temperature
    standard_name:                            unknown
    units:                                    K